In [20]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import matplotlib.cm as cm
import gc

# Parameters
radicals_feat_folder = "/app/notebooks/radical_features"
kanji_feat_folder = "/app/notebooks/kanji_features"
kanji_img_folder = "/app/data/jouyou"
output_dir = "/app/notebooks/kanji_radical_overlay"
PATCH_SIZE = 32
SIM_THRESHOLD = 0.90

os.makedirs(output_dir, exist_ok=True)
plt.ioff()

# Step 1: Load radical features
radical_features = {}
for fname in os.listdir(radicals_feat_folder):
    if fname.endswith(".npy"):
        radical_name = fname.replace(".npy", "")
        radical_features[radical_name] = np.load(os.path.join(radicals_feat_folder, fname))

radical_names = list(radical_features.keys())
radical_feats = np.stack([radical_features[name] for name in radical_names])  # (num_radicals, feat_dim)

# Step 2: Assign each radical a color
radical_to_color = {name: cm.tab20(i % 20) for i, name in enumerate(radical_names)}

# Step 3: Process each kanji
for fname in tqdm(os.listdir(kanji_feat_folder), desc="Visualizing kanji radicals"):
    if not fname.endswith(".npy"):
        continue

    kanji_name = fname.replace(".npy", "")
    feat_path = os.path.join(kanji_feat_folder, fname)
    img_path = os.path.join(kanji_img_folder, f"{kanji_name}.png")

    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        continue

    kanji_feat = np.load(feat_path)
    if kanji_feat.ndim != 2:
        print(f"Skipping {kanji_name}, bad shape: {kanji_feat.shape}")
        continue

    # Step 4: Cosine similarity between kanji patches and radicals
    sims = cosine_similarity(kanji_feat, radical_feats)  # (num_patches, num_radicals)

    matches = []
    for i, sim_vec in enumerate(sims):
        top_rad_idx = np.argmax(sim_vec)
        top_sim = sim_vec[top_rad_idx]
        if top_sim >= SIM_THRESHOLD:
            x = (i % 7) * PATCH_SIZE
            y = (i // 7) * PATCH_SIZE
            matches.append({
                "patch_coord": (x, y),
                "radical": radical_names[top_rad_idx],
                "similarity": top_sim
            })

    if not matches:
        continue  # skip kanji with no radical-level matches

    # Step 5: Load image and draw overlay
    img = Image.open(img_path).convert("L")
    fig, ax = plt.subplots()
    ax.imshow(img, cmap="gray")

    for m in matches:
        x, y = m["patch_coord"]
        radical = m["radical"]
        sim = m["similarity"]
        color = radical_to_color[radical]

        rect = patches.Rectangle((x, y), PATCH_SIZE, PATCH_SIZE,
                                 linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.3)
        ax.add_patch(rect)

        # Optional: show radical name
        # ax.text(x, y - 2, radical, fontsize=5, color='black', backgroundcolor='white')

    ax.set_title(f"Radical Overlay: {kanji_name}")
    ax.axis('off')

    out_path = os.path.join(output_dir, f"radical_overlay_{kanji_name}.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close(fig)

    del fig, ax, sims, matches, kanji_feat, img
    gc.collect()


Visualizing kanji radicals: 100%|█████████████████████████████████████████████████████████████| 2136/2136 [08:58<00:00,  3.97it/s]


In [ ]:
with torch.no_grad():
    # Forward pass and get attentions from last layer
    attentions = dino_model.get_last_selfattention(input_tensor)  # shape: (1, heads, 197, 197)


In [41]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from torchvision.utils import make_grid
import torch

# === Settings ===
kanji_name = "09664"  # <-- CHANGE THIS to any kanji .npy name
radicals_feat_folder = "/app/notebooks/radical_features"
kanji_feat_folder = "/app/notebooks/kanji_features"
kanji_img_folder = "/app/data/jouyou"
output_dir = "/app/notebooks/"
PATCH_SIZE = 32
SIM_THRESHOLD = 0.90

# Load features
radical_features = {f.replace(".npy", ""): np.load(os.path.join(radicals_feat_folder, f))
                    for f in os.listdir(radicals_feat_folder) if f.endswith(".npy")}
radical_names = list(radical_features.keys())
radical_feats = np.stack([radical_features[name] for name in radical_names])

kanji_feat = np.load(os.path.join(kanji_feat_folder, f"{kanji_name}.npy"))
sims = cosine_similarity(kanji_feat, radical_feats)

matches = []
for i, sim_vec in enumerate(sims):
    top_rad_idx = np.argmax(sim_vec)
    top_sim = sim_vec[top_rad_idx]
    if top_sim >= SIM_THRESHOLD:
        x = (i % 7) * PATCH_SIZE
        y = (i // 7) * PATCH_SIZE
        matches.append({
            "patch_coord": (x, y),
            "radical": radical_names[top_rad_idx],
            "similarity": top_sim
        })

# Load original kanji image
img_path = os.path.join(kanji_img_folder, f"{kanji_name}.png")
img = Image.open(img_path).convert("L")
img_np = np.array(img)

# === Extract & Save patch grid ===
patch_tensors = []
for m in matches[:5]:
    x, y = m["patch_coord"]
    patch = img_np[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
    tensor = torch.tensor(patch).unsqueeze(0).float() / 255.0
    patch_tensors.append(tensor)

if patch_tensors:
    grid = make_grid(torch.stack(patch_tensors), nrow=5, padding=2)
    plt.figure(figsize=(5, 1.5))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    plt.title(f"Top Patches: {kanji_name}")
    plt.savefig(os.path.join(output_dir, f"{kanji_name}_patch_grid.png"), bbox_inches="tight", dpi=120)
    plt.close()
    print(f"✅ Saved: {kanji_name}_patch_grid.png")
else:
    print("No patches above threshold found.")


✅ Saved: 09664_patch_grid.png


In [42]:
# Global patch bank
patch_bank = []

# Inside your kanji loop, after extracting matches[:5]
for m in matches[:5]:
    x, y = m["patch_coord"]
    patch = np.array(img)[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
    patch_bank.append(patch)

# After loop ends
np.save("/app/notebooks/patch_bank.npy", patch_bank)  # save for later


In [43]:
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np

patches = np.load("/app/notebooks/patch_bank.npy", allow_pickle=True)

# Pick 15 example patches
sampled = patches[:15]  # or use clustering/sorting later

# Convert to tensor
patch_tensors = [torch.tensor(p).unsqueeze(0).float() / 255.0 for p in sampled]
grid = make_grid(torch.stack(patch_tensors), nrow=5, padding=2)

# Save grid
plt.figure(figsize=(6, 2))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title("Micro-Pattern Patch Consistency")
plt.savefig("/app/notebooks/kanji_visualizations/micro_patterns.png", bbox_inches="tight", dpi=120)
plt.close()
